# PyMIF notebook 01 - read microscope datasets

This notebook is a set of practical templates for real microscope files. It does not ship real microscope data; fill in your own paths and run the matching section.

Every reader returns a manager with the same interface: `data`, `metadata`, `build_pyramid`, `to_zarr`, `visualize`, `subset_dataset`, and `update_metadata`.

In [ ]:
from pathlib import Path
import shutil
import sys
import numpy as np

# Use the installed package when available. When running from a local PyMIF
# source checkout, this fallback adds the repository root to sys.path.
try:
    import pymif.microscope_manager as mm
except ModuleNotFoundError:
    for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (candidate / "pymif").is_dir():
            sys.path.insert(0, str(candidate))
            break
    import pymif.microscope_manager as mm

OUTPUT_DIR = Path("pymif_tutorial_output")
OUTPUT_DIR.mkdir(exist_ok=True)
print("PyMIF managers loaded from:", mm.__file__)
print("Tutorial output folder:", OUTPUT_DIR.resolve())

In [ ]:
def summarize_manager(manager, title="dataset"):
    """Print the fields beginners usually need to inspect first."""
    print(f"\n{title}")
    print("-" * len(title))
    print("manager:", type(manager).__name__)
    print("axes:", manager.metadata.get("axes"))
    print("data_type:", manager.metadata.get("data_type", "intensity"))
    print("levels:", len(manager.data))
    for i, arr in enumerate(manager.data):
        print(f"  level {i}: shape={arr.shape}, chunks={getattr(arr, 'chunksize', None)}, dtype={arr.dtype}")
    for key in ["scales", "units", "time_increment", "time_increment_unit", "channel_names", "channel_colors"]:
        print(f"{key}:", manager.metadata.get(key))


def clean_path(path):
    path = Path(path)
    if path.exists():
        shutil.rmtree(path)
    return path

## One small helper for real-file examples

The helper below prints useful information without accidentally computing a large image. Dask arrays stay lazy until you call `.compute()`.

In [ ]:
def inspect_reader(manager):
    summarize_manager(manager, "loaded microscope dataset")
    print("\nFirst level object:", manager.data[0])
    print("A tiny lazy slice:", manager.data[0][tuple(slice(0, 1) for _ in range(manager.data[0].ndim))])

## Luxendo

Expected input: a Luxendo dataset folder containing XML metadata and `.lux.h5` files.

The reader produces TCZYX arrays.

In [ ]:
LUXENDO_PATH = None  # Example: "/data/sample_luxendo_folder"

if LUXENDO_PATH is not None:
    luxendo = mm.LuxendoManager(LUXENDO_PATH, chunks=(1, 1, 8, 1024, 1024))
    inspect_reader(luxendo)
else:
    print("Set LUXENDO_PATH to run this example.")

## Viventis

Expected input: a folder containing a companion `.ome` file and TIFF planes referenced by that metadata.

In [ ]:
VIVENTIS_PATH = None  # Example: "/data/viventis_export"

if VIVENTIS_PATH is not None:
    viventis = mm.ViventisManager(VIVENTIS_PATH, chunks=(1, 1, 8, 1024, 1024))
    inspect_reader(viventis)
else:
    print("Set VIVENTIS_PATH to run this example.")

## Opera / PerkinElmer Opera Phenix

Expected input: a pyramidal OME-TIFF file. The reader extracts embedded OME-XML and exposes the pyramid as Dask arrays.

In [ ]:
OPERA_PATH = None  # Example: "/data/opera/sample.ome.tif"

if OPERA_PATH is not None:
    opera = mm.OperaManager(OPERA_PATH, chunks=(1, 1, 8, 1024, 1024))
    inspect_reader(opera)
else:
    print("Set OPERA_PATH to run this example.")

## Zeiss CZI

Expected input: a `.czi` file. CZI files can contain multiple scenes. Use `scene_index` or `scene_name`.

In [ ]:
CZI_PATH = None  # Example: "/data/experiment.czi"
SCENE_INDEX = 0

if CZI_PATH is not None:
    zeiss = mm.ZeissManager(CZI_PATH, scene_index=SCENE_INDEX)
    print("Available scenes:", zeiss.scenes)
    inspect_reader(zeiss)
else:
    print("Set CZI_PATH to run this example.")

## SCAPE

Expected input: a Leica/SCAPE OME-TIFF with a sibling `Metadata` folder containing the matching `.xlif` file.

Use the argument name `ome_tiff_path` in notebook code.

In [ ]:
SCAPE_OME_TIFF_PATH = None  # Example: "/data/scape/p1.ome.tif"

if SCAPE_OME_TIFF_PATH is not None:
    scape = mm.ScapeManager(ome_tiff_path=SCAPE_OME_TIFF_PATH, chunks=(1, 1, 8, 1024, 1024))
    inspect_reader(scape)
else:
    print("Set SCAPE_OME_TIFF_PATH to run this example.")

## Convert any loaded reader to OME-Zarr

Once a microscope manager is loaded, conversion is the same for all readers.

For large files, choose chunk sizes carefully and build enough pyramid levels for visualization.

In [ ]:
# Replace this with whichever manager you loaded above, for example:
# manager = zeiss
manager = None

if manager is not None:
    manager.update_metadata({
        # Optional, but useful when channel names from the microscope are not clear.
        # "channel_names": ["DAPI", "GFP", "RFP"],
        # "channel_colors": ["0000FF", "00FF00", "FF0000"],
    })

    # Build a pyramid if the reader only supplied one level.
    # Use (1, 2, 2) to preserve Z spacing while downsampling Y and X.
    manager.build_pyramid(num_levels=3, downscale_factor=(1, 2, 2))

    output_zarr = clean_path(OUTPUT_DIR / "converted_from_microscope.zarr")
    manager.to_zarr(output_zarr, ngff_version="0.5", zarr_format=3)
    print("Wrote", output_zarr)
else:
    print("Load one of the readers above, then assign it to `manager`.")

## Read an existing Zarr

Use `ZarrManager` for both NGFF v0.4 and v0.5 stores. Use `ZarrV04Manager` only when you deliberately want the old compatibility wrapper.

In [ ]:
ZARR_PATH = None  # Example: "/data/my_dataset.zarr"

if ZARR_PATH is not None:
    z = mm.ZarrManager(ZARR_PATH, mode="r")
    inspect_reader(z)
    print("Image groups:", list(z.groups.keys()))
    print("Label groups:", list(z.labels.keys()))
else:
    print("Set ZARR_PATH to run this example.")